# Real-Time SMS Spam Detection

**Group Members**
- Abdul Rafay (B24S0348AI074)
- Farooq Awan (B24S0960AI065)

**Instructor:** Mr Waqas Yousaf — Machine Learning Lab

---

This notebook covers the full pipeline:
1. Data loading & exploration
2. Text preprocessing (lowercase → remove punctuation/numbers → stopword removal)
3. TF-IDF vectorization
4. Model training: Multinomial Naive Bayes + Linear SVM
5. Evaluation: Precision / Recall / F1 / Confusion Matrix
6. Saving the best model and vectorizer

## 0. Setup — Install / Import Libraries

In [ ]:
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# Download required NLTK data (runs once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

STOP_WORDS = set(stopwords.words('english'))

print('All imports successful.')

## 1. Data Loading

Dataset: **UCI SMS Spam Collection** (`data/spam.csv`)

Download from: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset  
Place the file at `data/spam.csv` before running this cell.

In [ ]:
# The raw CSV uses latin-1 encoding and has extra unnamed columns
df = pd.read_csv(
    'data/spam.csv',
    encoding='latin-1',
    usecols=[0, 1],           # keep only the label and text columns
    names=['label', 'text'],  # rename for clarity
    header=0,                 # skip original header row
)

print(f'Dataset shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
print('Class distribution:')
print(df['label'].value_counts())
print()
print(df['label'].value_counts(normalize=True).map('{:.1%}'.format))

In [ ]:
# Bar chart of class counts
ax = df['label'].value_counts().plot(
    kind='bar', color=['steelblue', 'tomato'], figsize=(6, 4), rot=0
)
ax.set_title('Message Class Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + 0.35, p.get_height() + 10))
plt.tight_layout()
plt.show()

In [ ]:
# Compare message length between ham and spam
df['msg_length'] = df['text'].str.len()

df.groupby('label')['msg_length'].describe()

In [ ]:
# Message length distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, (label, group) in zip(axes, df.groupby('label')):
    group['msg_length'].hist(bins=40, ax=ax, color='steelblue' if label == 'ham' else 'tomato')
    ax.set_title(f'Message length — {label}')
    ax.set_xlabel('Character count')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## 3. Text Preprocessing

Pipeline (applied in this order):
1. **Lowercase** — normalises word case
2. **Remove punctuation & numbers** — strip characters that carry no semantic value
3. **Tokenize** — split into individual words
4. **Remove stopwords** — filter common English words ('the', 'is', 'and', …)

In [ ]:
def clean_text(text: str) -> str:
    """Return a cleaned, stopword-free string ready for TF-IDF."""
    # Step 1: lowercase
    text = text.lower()
    # Step 2: remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Step 3: tokenize
    tokens = word_tokenize(text)
    # Step 4: remove stopwords and very short tokens (length <= 1)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return ' '.join(tokens)


# Apply to entire dataset
df['cleaned_text'] = df['text'].apply(clean_text)

# Encode labels: ham=0, spam=1
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})

print('Sample cleaned messages:')
df[['text', 'cleaned_text', 'label']].head(5)

## 4. Train / Test Split

- 80 % training / 20 % testing
- `stratify=y` preserves the ham/spam ratio in both splits (important for imbalanced data)

In [ ]:
X = df['cleaned_text']
y = df['label_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f'Training set: {len(X_train)} messages')
print(f'Test set:     {len(X_test)} messages')

## 5. TF-IDF Vectorization

| Parameter | Value | Reason |
|-----------|-------|--------|
| `ngram_range` | `(1, 2)` | Captures bigrams like "free offer", "call now" |
| `min_df` | `2` | Ignores terms appearing in fewer than 2 documents |
| `max_features` | `5000` | Caps vocabulary to keep memory low |
| `sublinear_tf` | `True` | Log-normalises term frequencies |

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=5000,
    sublinear_tf=True,
)

# Fit ONLY on training data — prevents data leakage
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')
print(f'Training matrix shape: {X_train_tfidf.shape}')

## 6. Model Training

We train and compare:
- **Multinomial Naive Bayes** — fast, probabilistic baseline for text classification
- **Linear SVM (LinearSVC)** — strong linear classifier for high-dimensional sparse data

In [ ]:
# --- Multinomial Naive Bayes ---
# alpha=0.1: tighter Laplace smoothing improves F1 on sparse TF-IDF features
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)

# --- Linear SVM ---
# C=1.0: default regularisation strength (good starting point)
# max_iter=2000: ensures convergence on larger datasets
svm_model = LinearSVC(C=1.0, max_iter=2000)
svm_model.fit(X_train_tfidf, y_train)

print('Both models trained.')

## 7. Evaluation

Focus on the **spam class** (label = 1) metrics because:
- **Precision** — avoid blocking legitimate messages (false positives)
- **Recall** — catch as much spam as possible (false negatives)
- **F1-score** — harmonic mean; use this to pick the better model

In [ ]:
def evaluate_model(name: str, model, X_test_vec, y_test):
    """Print classification report and plot confusion matrix."""
    preds = model.predict(X_test_vec)

    print(f'\n{'='*50}')
    print(f' {name}')
    print(f'{'='*50}')
    print(classification_report(y_test, preds, target_names=['Ham', 'Spam']))

    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt='d',
        xticklabels=['Ham', 'Spam'],
        yticklabels=['Ham', 'Spam'],
        cmap='Blues',
    )
    plt.title(f'Confusion Matrix — {name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

    return preds

In [ ]:
nb_preds  = evaluate_model('Multinomial Naive Bayes', nb_model,  X_test_tfidf, y_test)
svm_preds = evaluate_model('Linear SVM (LinearSVC)', svm_model, X_test_tfidf, y_test)

## 8. Select Best Model

In [ ]:
from sklearn.metrics import f1_score

nb_f1  = f1_score(y_test, nb_preds,  pos_label=1)
svm_f1 = f1_score(y_test, svm_preds, pos_label=1)

print(f'Naive Bayes  — Spam F1: {nb_f1:.4f}')
print(f'Linear SVM   — Spam F1: {svm_f1:.4f}')

if svm_f1 >= nb_f1:
    best_model      = svm_model
    best_model_name = 'Linear SVM'
else:
    best_model      = nb_model
    best_model_name = 'Multinomial Naive Bayes'

print(f'\n✅ Best model: {best_model_name} (Spam F1 = {max(nb_f1, svm_f1):.4f})')

## 9. Save Model & Vectorizer

Saved files are loaded by `app/app.py` for real-time inference.

In [ ]:
import os

os.makedirs('models', exist_ok=True)

joblib.dump(vectorizer,  'models/vectorizer.pkl')
joblib.dump(best_model,  'models/best_model.pkl')

print('Saved models/vectorizer.pkl')
print('Saved models/best_model.pkl')
print(f'Model type: {best_model_name}')

## 10. Quick Smoke Test

Verify the saved files work end-to-end before launching the Streamlit app.

In [ ]:
# Reload from disk
loaded_vectorizer = joblib.load('models/vectorizer.pkl')
loaded_model      = joblib.load('models/best_model.pkl')

def predict(message: str) -> str:
    """Return 'Spam' or 'Ham' for a raw SMS message."""
    cleaned  = clean_text(message)
    features = loaded_vectorizer.transform([cleaned])
    pred     = loaded_model.predict(features)[0]
    return 'Spam' if pred == 1 else 'Ham'


test_messages = [
    "Congratulations! You've won a FREE prize. Call now: 0800-123456",
    "Hey, are we still meeting at 6 pm tonight?",
    "URGENT: Your account has been suspended. Click here to verify.",
    "Don't forget to pick up milk on your way home.",
    "You have been selected for a cash reward of $1000. Reply WIN.",
]

for msg in test_messages:
    label = predict(msg)
    icon = '🔴' if label == 'Spam' else '🟢'
    print(f'{icon} [{label}] {msg[:70]}')

---

## Next Step

Launch the Streamlit web app:

```bash
streamlit run app/app.py
```

Open [http://localhost:8501](http://localhost:8501) in your browser.